# 03 — Benchmark : scoring, vérification, analyse des erreurs, exports

Dernière partie du pipeline :

7. Scoring **note vs recon** (officiel) et **source vs recon** (secondaire), en exact et en sémantique (LLM)
8. Vérification LLM de la cohérence note / reconstruction
9. Analyse des erreurs (type, étape, gravité) et cas difficiles par style de note
10. Synthèses, exports CSV, publication dans `results/` et figures dans `assets/figures/`

Pré-requis : notebook 02 exécuté (gold, notes et reconstructions présents sur disque).
Les scores sémantiques et la vérification utilisent MedGemma ; tous les appels LLM sont mis en cache.

In [ ]:
# Uniquement sur Google Colab : décommenter pour récupérer le dépôt et installer les dépendances
# !git clone https://github.com/<ton-compte>/bupa.git
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Rend `config` et `src` importables depuis notebooks/ (ou depuis la racine du dépôt)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

In [ ]:
# MedGemma est un modèle « gated » : il faut un token Hugging Face.
# En local : créer un fichier .env à la racine avec HF_TOKEN=hf_xxx (voir .env.example).
# Sur Colab : ajouter un secret nommé HF_TOKEN.
from src.llm import get_llm, hf_login

hf_login()
llm = get_llm()  # le modèle n'est réellement chargé qu'au premier appel
print("Model:", settings.MODEL_NAME, "| transformers/torch available:", llm.available())

## Chargement des sorties des étapes précédentes

In [ ]:
from src import gold, notes, reconstruction

patients = gold.load_gold_patients()
raw_notes, generated_notes = notes.load_notes()
recon_raw, recon_clean, recon_clipped = reconstruction.load_reconstructions()
print(f"{len(patients)} patients | {len(generated_notes)} notes | {len(recon_clean)} reconstructions")

## 10.1) Reset des sorties finales (optionnel)

Supprime uniquement scores / vérification / exports (pas le dataset filtré, le gold, les notes ni la reconstruction).
À faire juste avant un benchmark final pour repartir de sorties propres.

In [ ]:
from src import export

RESET_BENCHMARK_OUTPUTS = False
if RESET_BENCHMARK_OUTPUTS:
    removed = export.reset_benchmark_outputs()
    print("Removed files:", len(removed))
else:
    print("RESET_BENCHMARK_OUTPUTS = False -> no file removed.")

## 7) Scoring note vs recon et source vs recon

In [ ]:
from src import evaluation

df_scores = evaluation.score_all(patients, recon_clean, recon_raw, generated_notes, llm=llm)
display(df_scores)
evaluation.print_mean_scores(df_scores)

## 8) Chaîne de vérification LLM

In [ ]:
from src import verification

df_verification, df_verification_by_note_style = verification.run_verification(patients, recon_clean, generated_notes, llm=llm)
df_scores = verification.merge_verification_into_scores(df_scores, df_verification)
display(df_verification)
display(df_verification_by_note_style)

## 9) Analyse des erreurs, types d'erreurs et difficulté par `note_style`

Les scores disent **combien** la reconstruction est bonne ; l'analyse des erreurs dit **pourquoi**.
Pour chaque patient, et séparément pour les conditions et les medication requests :

- `overall` : type d'erreur dominant (`perfect_reconstruction`, `exact_wording_gap_only`, `source_to_note_gap_or_generation_loss`, `under_extraction_from_note`, ...)
- `stage` : étape probable du problème (`source_to_note`, `note_to_reconstruction`, `canonicalization_exact_matching`, `mixed`, `none`)
- `severity` : `none`, `mild`, `moderate`, `severe`
- `explanation` : résumé compact pour l'audit

In [ ]:
tables = evaluation.run_error_analysis(df_scores)
df_scores = tables["df_scores"]
display(tables["df_error_analysis"])
display(tables["df_error_types_by_note_style_conditions"])
display(tables["df_error_types_by_note_style_medication_requests"])
print("Hardest patients by note_style")
display(tables["df_hard_cases_by_note_style"])

## 9.1) Tables de synthèse

In [ ]:
df_scores_overall_summary, df_scores_summary_by_note_style = evaluation.run_summaries(df_scores, df_verification)
display(df_scores_overall_summary)
display(df_scores_summary_by_note_style)

## 11) Export des datasets depuis le cache

Relit tout le cache disque, reconstruit les datasets finaux, écrit les CSV dans `data/processed/exports/`
et une archive zip complète.

In [ ]:
exports = export.rebuild_datasets_from_cache()
df_export_review = export.export_datasets(exports)
display(df_export_review)

## 12) Publier dans `results/` et régénérer les figures

⚠️ Cette cellule **remplace** les CSV versionnés de `results/` et les figures de `assets/figures/`.
Ne l'active que pour un run final que tu veux publier.

In [ ]:
from src import plots

PUBLISH = False
if PUBLISH:
    written = export.publish_results()
    print(len(written), "CSV written to results/")
    for p in plots.make_all_figures():
        print("Figure:", p)

In [ ]:
# Inspecter un patient en détail
pid = exports["df_gold"].iloc[0]["patient_id"]
print(exports["df_notes"].set_index("patient_id").loc[pid, "note_text"])
exports["df_recon"].set_index("patient_id").loc[pid, ["raw_conditions", "conditions", "conditions_clipped"]].to_dict()